# IEEE-CIS Fraud Detection: Categorical-Level Submission
## DSCI 598 Capstone, Week 5
### Ben Miller

Companion notebook: `bm-fraud-categorical-tuning` (where the model type and settings below were chosen).

# Part 3: Submit the tuned model with categorical levels
> Submit the predictions generated by the optimal model found.

### Imports, seeds and constants

Paste the three `BEST_` values printed at the end of the tuning notebook. The feature and level settings must match the tuning notebook.

In [1]:
import os
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Kaggle mounts the competition files here (read-only)
DATA_DIR = '/kaggle/input/competitions/ieee-fraud-detection'

# Column names used throughout
ID_COLUMN = 'TransactionID'
TARGET = 'isFraud'
TIME_COLUMN = 'TransactionDT'      # seconds from an undisclosed starting point, not a real timestamp
AMOUNT_COLUMN = 'TransactionAmt'

# From the last code cell of bm-fraud-categorical-tuning
BEST_MODEL_NAME = 'lightgbm'
BEST_PARAMS = {'subsample': 0.8, 'reg_lambda': 5.0, 'num_leaves': 255, 'min_child_samples': 50, 'learning_rate': 0.02, 'colsample_bytree': 0.3}
BEST_N_ESTIMATORS = 418   # LightGBM only: average trees early stopping chose across the four folds (up from Week 4's 327, since the categorical levels give it more to learn)

# Numerical features (the decision above): measurement-like columns only
NUMERIC_FAMILIES = ['C', 'D', 'V', 'dist']   # counts, time deltas, Vesta features, distances
NUMERIC_IDENTITY_COLUMNS = [f'id_{number:02d}' for number in range(1, 12)]  # id_01-id_11 are measurements; id_12 on are categories
OTHER_NUMERIC_COLUMNS = [AMOUNT_COLUMN, 'has_identity']

# V-column reduction
CORRELATION_SAMPLE_SIZE = 100_000   # rows used to measure correlations (feature selection only, not tuning)
HIGH_CORRELATION = 0.95             # V columns more correlated than this with one already kept are dropped

# Model settings that aren't tuned
TREE_MISSING_VALUE = -999      # scikit-learn trees get missing values replaced by this; it sits below every real value
LOGREG_MAX_ITER = 2000
LIGHTGBM_MAX_ROUNDS = 3000     # upper limit; early stopping picks the real number
EARLY_STOPPING_ROUNDS = 100

# Categorical level selection. These rules are fit on training rows only.
MIN_LEVEL_COUNT = 500        # levels with fewer training transactions aren't tested; their fraud rate is too noisy to trust
FDR_ALPHA = 0.05             # Benjamini-Hochberg false discovery rate across every level tested
MIN_RATE_RATIO = 1.5         # a kept level's fraud rate is at least 1.5x the overall rate, or at most 1/1.5 of it
MAX_LEVELS_PER_COLUMN = 25   # strongest levels kept per column, so no single column adds hundreds of features
MISSING_LABEL = 'missing'    # missing values are a level of their own, since a blank field can be a signal
OTHER_LABEL = 'other'        # every level that isn't kept is grouped here, so no rows are lost

SUBMISSION_PATH = 'submission.csv'   # Kaggle offers files in /kaggle/working (the default folder) for submission

### Data-loading functions

In [2]:
def downcast_numeric_columns(frame):
    """
    Shrink float64 columns to float32 and integer columns to the smallest type that fits.
    The raw tables need several GB as float64; float32 roughly halves that, and tree
    models don't need the extra precision.
    """
    smaller_types = {}
    for column in frame.columns:
        if pd.api.types.is_float_dtype(frame[column]):
            smaller_types[column] = 'float32'
        elif pd.api.types.is_integer_dtype(frame[column]):
            smallest_fit = pd.to_numeric(frame[column], downcast='integer')
            smaller_types[column] = smallest_fit.dtype

    # one astype call for every column; assigning hundreds of columns one at a time fragments the DataFrame
    return frame.astype(smaller_types)


def standardize_identity_columns(identity):
    """
    Rename test_identity's 'id-01' style columns to train's 'id_01' spelling.
    The competition files use a hyphen in test and an underscore in train, so without
    this every identity feature would silently be missing in test after the merge.
    """
    renamed_columns = {}
    for column in identity.columns:
        if column.startswith('id-'):
            renamed_columns[column] = column.replace('id-', 'id_')
    return identity.rename(columns=renamed_columns)


def load_table(file_name):
    """One competition CSV, with its numeric columns downcast."""
    frame = pd.read_csv(os.path.join(DATA_DIR, file_name))
    return downcast_numeric_columns(frame)


def merge_identity(transactions, identity):
    """
    Left-join identity onto transactions by TransactionID, and flag which rows had a match.
    Only some transactions have identity data, so a left join keeps every transaction and
    leaves the identity columns missing for the rest.
    """
    # mark identity rows before the merge; transactions without a match end up missing, then 0
    flagged_identity = identity.assign(has_identity=1)
    merged = transactions.merge(flagged_identity, on=ID_COLUMN, how='left')
    merged['has_identity'] = merged['has_identity'].fillna(0).astype('int8')
    return merged


def load_split(split_name):
    """The merged transaction + identity table for 'train' or 'test'."""
    transactions = load_table(f'{split_name}_transaction.csv')
    identity = standardize_identity_columns(load_table(f'{split_name}_identity.csv'))
    return merge_identity(transactions, identity)


def memory_mb(frame):
    """How much memory a DataFrame uses, in megabytes."""
    return frame.memory_usage(deep=True).sum() / 1024 ** 2

### Load train and test

Train is sorted by time, as in the tuning notebook, so the V-column correlation sample picks the same rows.

In [3]:
train = load_split('train')
train = train.sort_values(TIME_COLUMN).reset_index(drop=True)
test = load_split('test')   # test keeps its original order, which the submission file has to match
print(f'train: {train.shape[0]:,} rows, test: {test.shape[0]:,} rows')

train: 590,540 rows, test: 506,691 rows


### Feature functions (from Week 4)

In [4]:
def column_family(column):
    """The feature family a column belongs to, from its name: 'V' for V1-V339, 'dist' for dist1-dist2, and so on."""
    if column.startswith('id_'):
        return 'id'
    return column.rstrip('0123456789')  # 'V258' -> 'V', 'dist1' -> 'dist'


def numeric_feature_columns(frame):
    """
    The measurement-like columns: amount, the identity flag, C, D, V, dist and id_01-id_11.
    Columns that hold category codes stored as numbers (card1-card6, addr1, addr2, id_12 on)
    are left out on purpose, along with the text columns.
    """
    features = []
    for column in frame.columns:
        if column_family(column) in NUMERIC_FAMILIES:
            features.append(column)
        elif column in NUMERIC_IDENTITY_COLUMNS or column in OTHER_NUMERIC_COLUMNS:
            features.append(column)
    return features


def select_uncorrelated_columns(frame, columns, threshold, sample_size):
    """
    Keep one column from each group of near-duplicates, scanning the columns in order.
    A column is dropped when its absolute correlation with a column already kept is above threshold.
    Correlations come from a random sample of rows: plenty to measure a correlation, and much faster than all rows.
    """
    sample = frame[columns].sample(n=min(sample_size, len(frame)), random_state=SEED)
    correlation = sample.corr().abs()
    kept_columns = []
    for column in columns:
        is_near_duplicate = False
        for kept_column in kept_columns:
            if correlation.loc[column, kept_column] > threshold:
                is_near_duplicate = True
                break
        if not is_near_duplicate:
            kept_columns.append(column)
    return kept_columns


def model_feature_columns(frame):
    """The numerical features with redundant V columns removed."""
    numeric_columns = numeric_feature_columns(frame)
    v_columns = []
    other_columns = []
    for column in numeric_columns:
        if column_family(column) == 'V':
            v_columns.append(column)
        else:
            other_columns.append(column)
    kept_v_columns = select_uncorrelated_columns(frame, v_columns, HIGH_CORRELATION, CORRELATION_SAMPLE_SIZE)
    return other_columns + kept_v_columns

### Categorical level functions

In [5]:
def categorical_feature_columns(frame, numeric_columns):
    """Every column that isn't numerical, the ID, the target or the raw time: the columns held back in Week 4."""
    excluded = set(numeric_columns) | {ID_COLUMN, TARGET, TIME_COLUMN}
    categorical_columns = []
    for column in frame.columns:
        if column not in excluded:
            categorical_columns.append(column)
    return categorical_columns


def categorical_labels(series):
    """
    One categorical column as text labels, with missing values labeled 'missing'.
    Number codes are converted to float first, so a code reads the same in train and test
    (1234.0 in both, not 1234 in one file and 1234.0 in the other).
    """
    is_missing = series.isna()
    if pd.api.types.is_numeric_dtype(series):
        series = series.astype('float64')
    labels = series.astype(str)
    labels[is_missing] = MISSING_LABEL
    return labels.astype('category')


def categorical_label_frame(frame, categorical_columns):
    """
    Text labels for every categorical column, built once up front.
    This step doesn't look at the target, so building it on all rows can't leak anything.
    """
    labels = {}
    for column in categorical_columns:
        labels[column] = categorical_labels(frame[column])
    return pd.DataFrame(labels, index=frame.index)


def test_levels(labels, target):
    """
    A chi-square test for every level of one column that has at least MIN_LEVEL_COUNT rows.
    Each test uses a 2x2 table (in this level or not, fraud or not) to ask whether the level's
    fraud rate differs from the rate in all the other rows.
    """
    target = np.asarray(target).astype('int64')   # isFraud is stored as int8, which would overflow in the counts below
    total_rows = len(target)
    total_fraud = int(target.sum())
    overall_rate = total_fraud / total_rows
    level_rows = labels.value_counts()
    level_fraud = pd.Series(target, index=labels.index).groupby(labels, observed=True).sum()

    rows = []
    for level, rows_in_level in level_rows.items():
        rows_in_level = int(rows_in_level)
        if rows_in_level < MIN_LEVEL_COUNT or rows_in_level == total_rows:
            continue  # too rare to trust, or the column has only one level here, so there's nothing to compare
        fraud_in_level = int(level_fraud[level])
        table = [
            [fraud_in_level, rows_in_level - fraud_in_level],
            [total_fraud - fraud_in_level, (total_rows - rows_in_level) - (total_fraud - fraud_in_level)],
        ]
        chi_square, p_value, _, _ = chi2_contingency(table)
        level_rate = fraud_in_level / rows_in_level
        rows.append({
            'level': level,
            'transactions': int(rows_in_level),
            'fraud_rate': level_rate,
            'rate_ratio': level_rate / overall_rate,
            'chi_square': chi_square,
            'p_value': p_value,
        })
    return pd.DataFrame(rows)


def find_predictive_levels(label_frame, target):
    """
    Test every level of every categorical column, then mark which levels to keep. A level is kept when it
    (1) is still significant after the Benjamini-Hochberg correction across all tests,
    (2) has a fraud rate at least MIN_RATE_RATIO times the overall rate, or at most 1/MIN_RATE_RATIO of it, and
    (3) ranks in its column's top MAX_LEVELS_PER_COLUMN by chi-square statistic.
    Returns one row per level tested.
    """
    column_reports = []
    for column in label_frame.columns:
        column_report = test_levels(label_frame[column], target)
        if len(column_report) > 0:
            column_report.insert(0, 'column', column)
            column_reports.append(column_report)
    if not column_reports:
        return pd.DataFrame(columns=['column', 'level', 'transactions', 'fraud_rate', 'rate_ratio',
                                     'chi_square', 'p_value', 'adjusted_p_value', 'kept'])

    report = pd.concat(column_reports, ignore_index=True)
    significant, adjusted_p_values, _, _ = multipletests(report['p_value'], alpha=FDR_ALPHA, method='fdr_bh')
    report['adjusted_p_value'] = adjusted_p_values
    report['significant'] = significant

    much_higher = report['rate_ratio'] >= MIN_RATE_RATIO
    much_lower = report['rate_ratio'] <= 1 / MIN_RATE_RATIO
    report['large_difference'] = much_higher | much_lower
    report['passes'] = report['significant'] & report['large_difference']

    # rank only the passing levels within each column; the rest get no rank and aren't kept
    passing = report[report['passes']]
    report['rank_in_column'] = passing.groupby('column')['chi_square'].rank(ascending=False, method='first')
    report['kept'] = report['passes'] & (report['rank_in_column'] <= MAX_LEVELS_PER_COLUMN)
    return report


def kept_levels_by_column(report):
    """The kept levels, grouped by column: {'ProductCD': ['C', 'W'], ...}."""
    kept_levels = {}
    kept_rows = report[report['kept']]
    for column, level in zip(kept_rows['column'], kept_rows['level']):
        if column not in kept_levels:
            kept_levels[column] = []
        kept_levels[column].append(level)
    return kept_levels


def safe_feature_name(text):
    """
    A column name LightGBM accepts. It rejects some punctuation in feature names, and level values like
    device names can contain it, so anything that isn't a letter, digit, dot or dash becomes an underscore.
    """
    characters = []
    for character in text:
        if character.isalnum() or character in '.-':
            characters.append(character)
        else:
            characters.append('_')
    return ''.join(characters)


def level_indicator_columns(label_frame, kept_levels):
    """
    A 0/1 column for each kept level, plus one 'other' column per categorical column for every level that
    wasn't kept. Columns with no kept levels are left out entirely.
    """
    indicators = {}
    for column, levels in kept_levels.items():
        labels = label_frame[column]
        in_a_kept_level = np.zeros(len(labels), dtype=bool)
        for level in levels:
            is_level = (labels == level).to_numpy()
            name = safe_feature_name(f'{column}__{level}')
            if name in indicators:
                name = f'{name}_{len(indicators)}'   # two levels that differ only in punctuation would otherwise share a name
            indicators[name] = is_level
            in_a_kept_level = in_a_kept_level | is_level
        indicators[safe_feature_name(f'{column}__{OTHER_LABEL}')] = np.logical_not(in_a_kept_level)
    return pd.DataFrame(indicators, index=label_frame.index).astype('float32')


def build_model_matrix(numeric_frame, label_frame, kept_levels):
    """The numerical features followed by the level indicator columns, all as float32."""
    indicators = level_indicator_columns(label_frame, kept_levels)
    return pd.concat([numeric_frame.astype('float32'), indicators], axis=1)

### Select levels on all training data and build both matrices

With no validation fold left, the level tests run once on all of the training data. Test's matrix uses the same kept levels and the same column order.

In [6]:
numeric_columns = numeric_feature_columns(train)
numeric_feature_names = model_feature_columns(train)
categorical_columns = categorical_feature_columns(train, numeric_columns)

train_labels = categorical_label_frame(train, categorical_columns)
test_labels = categorical_label_frame(test, categorical_columns)
report = find_predictive_levels(train_labels, train[TARGET].to_numpy())
kept_levels = kept_levels_by_column(report)

train_matrix = build_model_matrix(train[numeric_feature_names], train_labels, kept_levels)
test_matrix = build_model_matrix(test[numeric_feature_names], test_labels, kept_levels)

print(f"{len(numeric_feature_names)} numerical features, {int(report['kept'].sum())} levels kept from {len(kept_levels)} columns")
print(f'{train_matrix.shape[1]} features in total; train and test columns match: {list(train_matrix.columns) == list(test_matrix.columns)}')

272 numerical features, 318 levels kept from 48 columns
638 features in total; train and test columns match: True


### Model builders (from Week 4)

In [7]:
def build_logistic_regression(params):
    """Median imputation and scaling, since logistic regression handles neither missing values nor mixed scales."""
    return make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        LogisticRegression(max_iter=LOGREG_MAX_ITER, **params),   # ** passes each tuned setting as its own argument
    )


def build_decision_tree(params):
    """A single decision tree, with missing values replaced by a stand-in value."""
    return make_pipeline(
        SimpleImputer(strategy='constant', fill_value=TREE_MISSING_VALUE),
        DecisionTreeClassifier(random_state=SEED, **params),
    )


def build_random_forest(params):
    """A random forest, with missing values replaced by a stand-in value. n_jobs=-1 uses every CPU core."""
    return make_pipeline(
        SimpleImputer(strategy='constant', fill_value=TREE_MISSING_VALUE),
        RandomForestClassifier(random_state=SEED, n_jobs=-1, **params),
    )


def build_lightgbm(params):
    """LightGBM handles missing values itself. Its tree count is a ceiling that early stopping cuts short."""
    return lgb.LGBMClassifier(
        n_estimators=LIGHTGBM_MAX_ROUNDS,
        subsample_freq=1,      # subsample only takes effect when this is 1 or more
        metric='auc',          # early stopping watches AUC, not the default log loss
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
        **params,
    )


MODEL_BUILDERS = {
    'logistic_regression': build_logistic_regression,
    'decision_tree': build_decision_tree,
    'random_forest': build_random_forest,
    'lightgbm': build_lightgbm,
}

### Train on all training data and write the submission

For LightGBM there's no validation slice left for early stopping, so it uses the average number of trees early stopping chose during tuning.

In [8]:
def build_final_model(model_name, params, n_estimators):
    """The chosen model type with its tuned settings. LightGBM gets a fixed tree count instead of early stopping."""
    model = MODEL_BUILDERS[model_name](params)
    if model_name == 'lightgbm':
        model.set_params(n_estimators=n_estimators)
    return model


final_model = build_final_model(BEST_MODEL_NAME, BEST_PARAMS, BEST_N_ESTIMATORS)
final_model.fit(train_matrix, train[TARGET])
test_fraud_probabilities = final_model.predict_proba(test_matrix)[:, 1]

submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN], TARGET: test_fraud_probabilities})
submission.to_csv(SUBMISSION_PATH, index=False)

# The submission must have one row per test transaction, in the same order as Kaggle's sample file
sample_submission = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
same_ids = np.array_equal(submission[ID_COLUMN].to_numpy(), sample_submission[ID_COLUMN].to_numpy())
print(f'wrote {SUBMISSION_PATH}: {len(submission):,} rows; matches sample_submission IDs: {same_ids}')
print(submission.head().to_string(index=False))

wrote submission.csv: 506,691 rows; matches sample_submission IDs: True
 TransactionID  isFraud
       3663549 0.003530
       3663550 0.002761
       3663551 0.006038
       3663552 0.002865
       3663553 0.003327


### Kaggle score

- Model: **LightGBM** with 272 numerical features plus **318** categorical levels (638 features in total, counting one "other" column for each of the 48 columns with kept levels).
- Public score **0.9326**, private score **0.9086** (Week 4: 0.8936 private, 0.9115 public).
- Compared with the tuning notebook's last-fold AUC of **0.9126**: the private score is 0.0040 lower, close to Week 4's gap of 0.0024, so the time-ordered folds are still a good estimate of how the model does on future data. The private score went up 0.0150 from Week 4.